# Monosphere プリセット生成 Round 3（カバレッジ全踏破）

プール151件時点で未踏だった Layer2 の組み合わせ475通りを1件ずつ生成し、Layer3 の未踏162通りも同じプリセットに相乗りさせて、両レイヤーの組み合わせを全踏破します。点火14系統は均等ローテーション。

In [ ]:
# Monosphere プリセット生成 Round 3: カバレッジ全踏破（2026-11-09）
# 使い方: frontier / showcase / generated R1・R2 の各JSON と parameterCatalog.ts を同じ場所に置いて実行。
import json
bundle = {"presets": [], "catalog": open("parameterCatalog.ts").read()}
for fn in ("frontier-presets-20261107.json", "showcase-presets-20261108.json",
           "generated-presets-20261109.json", "generated-presets-20261109-r2.json"):
    d = json.load(open(fn))
    bundle["presets"] += [{"id": p["id"], "name": p["name"], "config": p["config"]} for p in d["presets"]]
import gzip, base64
open("/tmp/g.b64", "w").write(base64.b64encode(gzip.compress(json.dumps(bundle).encode())).decode())


In [ ]:
bundle = json.loads(gzip.decompress(base64.b64decode(open("/tmp/g.b64").read())))
pool = bundle["presets"]
catalog_text = bundle["catalog"]
print("pool:", len(pool))

ranges = {}
for m in re.finditer(r"key: '([A-Za-z0-9]+)'[^}]*?min: ([-\d.e+]+), max: ([-\d.e+]+)", catalog_text):
    ranges[m.group(1)] = (float(m.group(2)), float(m.group(3)))

union_keys = sorted(set().union(*[set(p["config"]) for p in pool]))
reference = {}
for k in union_keys:
    for p in pool:
        if k in p["config"]:
            reference[k] = p["config"][k]
            break
int_keys = set()
for k in union_keys:
    vals = [p["config"][k] for p in pool if k in p["config"]]
    if vals and all(isinstance(v, (int, float)) and not isinstance(v, bool) and float(v).is_integer() for v in vals):
        int_keys.add(k)

def corpus_range(k):
    vals = [float(p["config"][k]) for p in pool
            if isinstance(p["config"].get(k), (int, float)) and not isinstance(p["config"].get(k), bool)]
    return (min(vals), max(vals)) if vals else None

def value_range(k):
    return ranges.get(k) or corpus_range(k)

ENUM_KEYS = ["layer2Source", "layer2Type", "layer3Source", "layer3Type",
             "layer2TemporalProfile", "layer3TemporalProfile",
             "layer2MaterialStyle", "layer3MaterialStyle"]
vocab = {k: sorted({str(p["config"][k]) for p in pool if k in p["config"]}) for k in ENUM_KEYS}
vocab["gpgpuAttractorType"] = ["lorenz", "rossler", "thomas"]
vocab["gpgpuVFieldType"] = ["dipole", "saddle", "source", "spiral"]
vocab["gpgpuSdfShape"] = ["box", "capsule", "sphere", "torus"]
vocab["gpgpuEmitShape"] = ["box", "cone", "disc", "ring", "shell", "sphere"]

def combos(layer):
    return {(str(p["config"][f"{layer}Source"]), str(p["config"][f"{layer}Type"]))
            for p in pool if f"{layer}Source" in p["config"] and f"{layer}Type" in p["config"]}

present2, present3 = combos("layer2"), combos("layer3")
missing2 = [(s, t) for s in vocab["layer2Source"] for t in vocab["layer2Type"] if (s, t) not in present2]
missing3 = [(s, t) for s in vocab["layer3Source"] for t in vocab["layer3Type"] if (s, t) not in present3]
py_rng.shuffle(missing2)
py_rng.shuffle(missing3)
grid2 = len(vocab["layer2Source"]) * len(vocab["layer2Type"])
grid3 = len(vocab["layer3Source"]) * len(vocab["layer3Type"])
print(f"L2: present {len(present2)}/{grid2}, missing {len(missing2)}")
print(f"L3: present {len(present3)}/{grid3}, missing {len(missing3)}")

GROUPS = [
    ("Fluid", "gpgpuFluidEnabled", "gpgpuFluid"), ("SPH", "gpgpuSphEnabled", "gpgpuSph"),
    ("Spring", "gpgpuSpringEnabled", "gpgpuSpring"), ("Verlet", "gpgpuVerletEnabled", "gpgpuVerlet"),
    ("Elastic", "gpgpuElasticEnabled", "gpgpuElastic"), ("Magnetic", "gpgpuMagneticEnabled", "gpgpuMagnetic"),
    ("Metaball", "gpgpuMetaballEnabled", "gpgpuMetaball"), ("SDF Collider", "gpgpuSdfEnabled", "gpgpuSdf"),
    ("Tube", "gpgpuTubeEnabled", "gpgpuTube"), ("Volumetric", "gpgpuVolumetricEnabled", "gpgpuVolumetric"),
    ("Ambient", "ambientEnabled", "ambient"), ("InterLayer Collision", "interLayerCollisionEnabled", "interLayer"),
    ("Attractor Swap", None, "gpgpuAttractor"), ("VField Swap", None, "gpgpuVField"),
]

def jitter_value(k):
    r = value_range(k)
    if r is None or r[1] <= r[0]:
        return None
    nv = r[0] + py_rng.random() * (r[1] - r[0])
    return int(round(nv)) if k in int_keys else nv

def ignite(cfg, group):
    title, toggle, stem = group
    if title == "Attractor Swap":
        cfg["gpgpuEnabled"] = True; cfg["gpgpuAttractorEnabled"] = True
        cfg["gpgpuAttractorType"] = py_rng.choice(["rossler", "thomas"])
    elif title == "VField Swap":
        cfg["gpgpuEnabled"] = True
        if "gpgpuVFieldEnabled" in cfg: cfg["gpgpuVFieldEnabled"] = True
        cfg["gpgpuVFieldType"] = py_rng.choice(["dipole", "saddle", "source"])
    else:
        cfg[toggle] = True
        if toggle.startswith("gpgpu"): cfg["gpgpuEnabled"] = True
    for k in union_keys:
        if k.startswith(stem) and k != toggle and k not in ("gpgpuAttractorType", "gpgpuVFieldType"):
            v = cfg.get(k)
            if isinstance(v, (int, float)) and not isinstance(v, bool) and py_rng.random() < 0.7:
                nv = jitter_value(k)
                if nv is not None: cfg[k] = nv
    return title

def mutate(cfg, frac):
    keys = [k for k in union_keys if isinstance(cfg.get(k), (int, float)) and not isinstance(cfg.get(k), bool)]
    for k in py_rng.sample(keys, max(1, int(len(keys) * frac))):
        r = value_range(k)
        if r is None or r[1] <= r[0]: continue
        span = (r[1] - r[0]) * 0.25
        nv = min(r[1], max(r[0], float(cfg[k]) + py_rng.uniform(-span, span)))
        cfg[k] = int(round(nv)) if k in int_keys else nv

def titleize(s):
    return " ".join(w.capitalize() for w in str(s).replace("_", " ").split())

now = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.000Z")
used_names = {p["name"] for p in pool}
presets_out, group_counts, donor_counts = [], Counter(), Counter()
for idx, (s2, t2) in enumerate(missing2, 1):
    donor = py_rng.choice(pool)
    donor_counts[donor["name"]] += 1
    cfg = {k: (donor["config"][k] if k in donor["config"] else reference[k]) for k in union_keys}
    cfg["layer2Source"], cfg["layer2Type"] = s2, t2
    l3_note = ""
    if idx - 1 < len(missing3):
        s3, t3 = missing3[idx - 1]
        cfg["layer3Source"], cfg["layer3Type"] = s3, t3
        l3_note = f" / L3 {s3} x {t3}"
    applied = []
    if idx % 5 != 4:
        g = GROUPS[idx % len(GROUPS)]
        applied.append(ignite(cfg, g))
        group_counts[g[0]] += 1
    mutate(cfg, 0.05)
    for k in ("layer2TemporalProfile", "layer3TemporalProfile", "layer2MaterialStyle", "layer3MaterialStyle"):
        if k in vocab and py_rng.random() < 0.3:
            cfg[k] = py_rng.choice(vocab[k])
    name = f"Coverage {idx:03d} — {titleize(s2)} × {titleize(t2)}"
    if name in used_names: name += " II"
    used_names.add(name)
    slug = re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")
    presets_out.append({"id": f"{slug}-20261109", "name": name, "config": cfg,
                        "createdAt": now, "updatedAt": now})
    if idx <= 3 or idx == len(missing2):
        print(f"  {name} | donor: {donor['name']} | groups: {applied}{l3_note}")

# validation
errors = []
ref_types = {k: type(reference[k]).__name__ for k in union_keys}
new_combos2 = {(p["config"]["layer2Source"], p["config"]["layer2Type"]) for p in presets_out}
new_combos3 = {(p["config"]["layer3Source"], p["config"]["layer3Type"]) for p in presets_out}
for p in presets_out:
    if set(p["config"]) != set(union_keys): errors.append(f"{p['name']}: keyset")
    for k, v in p["config"].items():
        rt = ref_types[k]
        if rt == "bool" and not isinstance(v, bool): errors.append(f"{p['name']}:{k}")
        elif rt in ("int", "float") and (not isinstance(v, (int, float)) or isinstance(v, bool) or not math.isfinite(float(v))): errors.append(f"{p['name']}:{k}")
        elif rt == "str" and not isinstance(v, str): errors.append(f"{p['name']}:{k}")
print("validation errors:", len(errors), errors[:5])
print(f"L2 after: {len(present2 | new_combos2)}/{grid2} | L3 after: {len(present3 | new_combos3)}/{grid3}")

parts = []
for start in range(0, len(presets_out), PART_SIZE):
    chunk = presets_out[start:start + PART_SIZE]
    parts.append({"version": 1, "activePresetId": chunk[0]["id"], "currentConfig": chunk[0]["config"],
                  "exportedAt": now, "presetBlendDuration": 1.4, "presetSequence": [],
                  "presets": chunk, "sequenceLoopEnabled": True})
blob = base64.b64encode(gzip.compress(json.dumps(parts).encode())).decode()
open("/tmp/parts.json.gz.b64", "w").write(blob)
report = {"total_generated": len(presets_out), "parts": len(parts),
          "l2": {"grid": grid2, "before": len(present2), "after": len(present2 | new_combos2)},
          "l3": {"grid": grid3, "before": len(present3), "after": len(present3 | new_combos3)},
          "group_counts": dict(group_counts), "validation_errors": errors,
          "top_donors": donor_counts.most_common(5)}
json.dump(report, open("/tmp/coverage_report.json", "w"), ensure_ascii=False)
print("parts:", len(parts), "| blob chars:", len(blob))
print("DONE")
